# Faithful Explanations

A three-role interpretability audit of a frozen DistilBERT sentiment classifier. You will extract, defend, and cross-check the token-level rationales behind 100 of its predictions, and hand in three CSVs, one per colleague.

> This is the **starter** notebook: the scaffolding, the loaders, and the exact local evaluators are all here and correct. Each part ships with a **simple baseline that runs top-to-bottom**, plus a clearly marked `YOUR CODE HERE` block where you write the real strategy. Run it once end to end, confirm the three CSVs land in `out/`, then iterate.

## **The story**

You're an ML interpretability researcher. Six months ago a streaming platform was sued over moderation decisions made by an automated sentiment classifier, a fine-tuned DistilBERT-SST2 that flags user reviews as positive or negative. The court ordered an expert audit and your firm took the contract. Three colleagues have spent weeks pulling 100 specific reviews from the case file. They are sentences where the classifier was confident, sometimes wrong, sometimes consequentially right. They've handed the file to you. The classifier is opaque, you're not allowed to retrain it, and you have to give each colleague something they can put in front of the court.

The **first colleague** is the technical witness for the plaintiffs. She needs to argue, sentence by sentence, that the classifier's decision was driven by specific tokens in the input. Her notion of *driven* is concrete and unforgiving: take the tokens you nominate as the rationale, mask them out of the sentence, and the model's confidence in its original verdict has to collapse. Conversely, mask everything *except* those tokens and the confidence has to hold up. She doesn't care whether your nomination matches human intuition. She cares whether it would survive cross-examination by a competent ML engineer.

The **second colleague** is a safety researcher running the defense's counter-argument. He suspects the classifier's reasoning is fragile, that any given verdict has only one piece of evidence holding it up, and his goal is to argue the opposite: that for each sentence, the model's decision is supported by *two genuinely different* faithful rationales. If you can find such pairs, the model's behavior is robust. If you can only find one rationale per sentence, his case falls apart. He'll only accept pairs that pass the first witness's probe on *both* slots, and that share as little token content as possible. Same mask twice doesn't count.

The **third colleague** is the journalist who broke the story. She doesn't trust either of the technical witnesses, and she's not interested in what the model 'actually computes' on. She wants to know what *people* would point to as the meaningful words. She has a panel of annotators and a hidden rationale set built from a consensus of attribution methods, and she'll score your rationales against that set. Her notion of a good rationale is the one a human would underline.

The three reviews are the three subtasks below: **1** (Witness), **2** (Counter-argument), and **3** (Journalist).

## **How do you find a faithful rationale**

Two ideas from post-hoc explainability cover everything you need.

The first idea is that **attribution methods rank input tokens by importance** to a prediction. Several methods exist. Attention rollout aggregates attention weights and tells you where the network looks. Occlusion replaces each token with `[MASK]` one at a time and measures the drop in the predicted-class logit. This is a literal counterfactual, a causal-effect estimate. Integrated gradients and gradient x input approximate the contribution of each token via Taylor-style expansions of the prediction. None of these methods agrees with the others on most sentences (Krishna et al. 2022). The disagreement is the door this task walks through.

The second idea is that **faithfulness probes test whether a candidate token-subset actually carries the model's decision**. The standard protocol is ERASER (DeYoung et al. 2020), which uses two probes: *sufficiency* checks that keeping only the candidate tokens preserves the prediction, *comprehensiveness* checks that removing them destroys it. A mask is faithful if it passes both. The probe is formal. You'll see it implemented in `faithful_verdict` below, and it is exactly what 'faithful' means in this task.

Each colleague rewards different choices. The first witness wants masks that pass the faithfulness probe. Occlusion is the natural attribution method here because it estimates the same causal effect the probe rewards. The second wants pairs of faithful masks that don't overlap. Off-the-shelf methods don't directly give that, you have to engineer it. The third doesn't run the model at all. She compares your tokens to a human-aligned set, which the consensus across multiple attribution methods approximates well.

## **Concepts you'll see along the way**

A few primitives the notebook leans on.

- **Attention rollout.** The transformer's last-layer attention from `[CLS]` to each input token, averaged over heads. Gives a per-token score that captures *where the model looks*. Famously misleading on its own: a token can be heavily attended to without driving the prediction (Jain & Wallace 2019).
- **Occlusion.** Replace each non-special token with `[MASK]` one at a time, measure how much the predicted-class logit drops. The largest drops mark the tokens the model causally relies on. The closest of the standard methods to a counterfactual, and the most aligned with the ERASER probe. Precomputed for you in `occlusion_pack.json`.
- **Integrated gradients (and gradient x input).** Both approximate each token's contribution via a Taylor-style expansion around the input embedding. Faster than occlusion (one forward + one backward pass) but biased toward tokens with large embedding norms.
- **ERASER's sufficiency and comprehensiveness probes.** Let $\Delta = \ell_{\hat{y}} - \ell_{1-\hat{y}}$ be the predicted-class logit margin. For a candidate mask $M$, faithfulness requires *both* $\Delta(\text{suff}(M)) \geq 0.5\,\Delta(\text{orig})$ and $\Delta(\text{orig}) - \Delta(\text{comp}(M)) \geq 0.3\,\Delta(\text{orig})$. `suff(M)` keeps the mask and `[MASK]`s everything else; `comp(M)` `[MASK]`s the mask.
- **Jaccard distance.** $1 - |A \cap B| / |A \cup B|$. 0 if A and B are identical, 1 if disjoint. The diversity factor in 2.
- **Binary F1.** Harmonic mean of precision and recall on the positive class. The metric 3 uses to compare your per-token mask against the hidden rationale set.

## **The submission format (read this once, it applies to all three subtasks)**

Every subtask is graded as a **static CSV** with three columns: `subtaskID,datapointID,answer`. The judge never runs your Python or the model, it just reads your CSV. `subtaskID` is `1` for every standalone submission. `answer` is a per-token **binary** label: `1` if the token is in your mask, `0` otherwise. You emit exactly one row per **content token** of each sentence (token indices `1 .. T-2`, where `[CLS]` at index 0 and the final `[SEP]` are excluded).

The `datapointID` encodes which (sentence, token) the row is about:

- **1** and **3**: `datapointID = sid * 1000 + tok`.
- **2**: `datapointID = sid * 2000 + slot * 1000 + tok`, with `slot` in `{0, 1}`.

The writers below build these rows for you from a plain `{sentence_id: mask}` dict, so you only ever think in terms of masks. Because the judge compares per-token labels, the mask cap still matters: submitting 'all tokens' tanks your precision.

## **Imports**

We pin a seed for reproducibility of your local exploration. The judge is deterministic and doesn't use it.

In [1]:
def print_dir(obj):
    for i in dir(obj):
        if i.startswith("_"):
            continue
        print(i)

In [2]:
import csv
import json
import os
import random
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cuda


## **The data (shipped with the task)**

The starter kit is bundled with the task under `./data`: the JSON packs plus the classifier itself in `./data/model`. Nothing is downloaded at runtime — the contest environment is offline.

- `sentences.json`, `classifier_info.json` : the 100 scored sentences + model/cap config.
- `occlusion_pack.json` : precomputed per-token occlusion deltas + `delta_ref` for the 100 sentences (so you don't have to run the model at all for a first pass).
- `practice_sentences.json`, `practice_rationales.json`, `practice_occlusion_pack.json` : 25 labeled practice sentences (not scored) with the same protocol, for your local 3 F1 self-check.
- `model/` : the bundled classifier, loaded offline with `local_files_only=True`.


In [3]:
from pathlib import Path

# The starter kit ships WITH the task under ./data (JSON packs + the bundled
# classifier in ./data/model). The contest environment is offline: nothing is
# downloaded at runtime.
DATA = Path("./data/")
REQUIRED = ["sentences.json", "classifier_info.json", "occlusion_pack.json",
            "practice_sentences.json", "practice_rationales.json"]
_missing = [f for f in REQUIRED if not (DATA / f).exists()]
assert not _missing, f"Missing starter-kit files under ./data: {_missing}"
assert (DATA / "model").is_dir(), "Bundled model folder ./data/model is missing"
print("Starter-kit files found under ./data (offline, no download needed).")


Starter-kit files found under ./data (offline, no download needed).


## **Load the model**

The classifier is `distilbert-base-uncased-finetuned-sst-2-english` from HuggingFace, frozen. Output: two logits per input, index 0 NEGATIVE, index 1 POSITIVE. We load with `attn_implementation="eager"` so `output_attentions=True` works if you want to build attention rollout for subtask 3. The model is only needed to *validate* your masks locally (the probe, self-checks). The judge doesn't run it.

In [4]:
with open(DATA / 'classifier_info.json') as f:
    info = json.load(f)
MODEL_NAME = info['model_name']
MAX_MASK_TOKENS = info['max_mask_tokens']
MAX_MASK_FRACTION = info['max_mask_fraction']

# Load the classifier from the bundled local copy -- fully offline, no HuggingFace download.
MODEL_DIR = DATA / "model"
tok = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR, attn_implementation='eager', local_files_only=True).to(DEVICE).eval()
MASK_TOKEN_ID = tok.mask_token_id

print(f'Classifier: {MODEL_NAME} (loaded offline from {MODEL_DIR})')
print(f'Mask cap:   min({MAX_MASK_TOKENS}, {MAX_MASK_FRACTION:.2f} * real_tokens) per sentence')


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Classifier: distilbert-base-uncased-finetuned-sst-2-english (loaded offline from data/model)
Mask cap:   min(12, 0.30 * real_tokens) per sentence


## **Load the data**

100 SST-2 dev sentences pre-tokenized with DistilBERT's WordPiece tokenizer. Each entry has the raw `text`, the `tokens` list (starting with `[CLS]`, ending with `[SEP]`), the integer `token_ids`, the `attention_mask`, the model's `predicted_label`, and its `confidence`. Your masks index into `token_ids`. Do not retokenize with anything else, the indices won't line up.

The mask cap is enforced per sentence: at most 12 tokens, or 30 percent of the real-token count, whichever is smaller. `cap_for_sentence` computes it. `content_tokens` gives the token indices you're allowed to nominate (everything but `[CLS]` and the final `[SEP]`), and is also exactly the set of rows each submission CSV must contain.

In [5]:
with open(DATA / 'sentences.json') as f:
    sentences = json.load(f)
print(f'Sentences: {len(sentences)}')
print(sentences[0])

def cap_for_sentence(s: dict) -> int:
    real = sum(s['attention_mask']) - 2
    return min(MAX_MASK_TOKENS, max(1, int(MAX_MASK_FRACTION * real)))

def content_tokens(s: dict) -> list:
    """Token indices you may nominate: 1 .. T-2 ([CLS] and final [SEP] excluded).
    Also the exact set of (per-token) rows each submission CSV must contain."""
    return list(range(1, sum(s['attention_mask']) - 1))

Sentences: 100
{'text': "determined to be fun , and bouncy , with energetic musicals , the humor did n't quite engage this adult .", 'tokens': ['[CLS]', 'determined', 'to', 'be', 'fun', ',', 'and', 'bo', '##un', '##cy', ',', 'with', 'energetic', 'musicals', ',', 'the', 'humor', 'did', 'n', "'", 't', 'quite', 'engage', 'this', 'adult', '.', '[SEP]'], 'token_ids': [101, 4340, 2000, 2022, 4569, 1010, 1998, 8945, 4609, 5666, 1010, 2007, 18114, 20103, 1010, 1996, 8562, 2106, 1050, 1005, 1056, 3243, 8526, 2023, 4639, 1012, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'predicted_label': 0, 'confidence': 0.9997, 'sentence_id': 0}


## **Eval helpers (this is what 'faithful' means, verbatim)**

The ERASER probe below is the ground truth of faithfulness for this task. Use `faithful_verdict` to check whether a mask is faithful (1, and to gate 2 pairs); `soft_faithful` is its continuous version; `jaccard_distance` is the diversity factor in 2. These run the frozen model, so they need your GPU.

- `faithful_verdict(s, mask_set) -> 0/1`: sufficiency AND comprehensiveness.
- `soft_faithful(s, mask_set) -> [0,1]`: partial credit below each threshold.
- `jaccard_distance(a, b)`: mask diversity.

In [6]:
ALPHA_SUFF = 0.5
TAU_COMP = 0.3

@torch.no_grad()
def forward_logits(ids, am):
    return model(input_ids=ids, attention_mask=am).logits.squeeze(0)

def mask_to_input(token_ids, attention_mask, mask_set, mode):
    """mode='suff': keep mask, [MASK] the rest. mode='comp': remove mask, [MASK] those positions."""
    out = list(token_ids)
    seq_len = sum(attention_mask)
    for i in range(1, seq_len - 1):
        in_mask = i in mask_set
        if mode == 'suff' and not in_mask:
            out[i] = MASK_TOKEN_ID
        elif mode == 'comp' and in_mask:
            out[i] = MASK_TOKEN_ID
    return out

def _deltas(s, mask_set):
    ids = torch.tensor([s['token_ids']], device=DEVICE)
    am = torch.tensor([s['attention_mask']], device=DEVICE)
    pred = s['predicted_label']
    ref = forward_logits(ids, am)
    delta_ref = float(ref[pred] - ref[1 - pred])
    suff_ids = torch.tensor([mask_to_input(s['token_ids'], s['attention_mask'], mask_set, 'suff')], device=DEVICE)
    comp_ids = torch.tensor([mask_to_input(s['token_ids'], s['attention_mask'], mask_set, 'comp')], device=DEVICE)
    # print(type(suff_ids), type(am))
    suff = forward_logits(suff_ids, am)
    comp = forward_logits(comp_ids, am)
    delta_suff = float(suff[pred] - suff[1 - pred])
    delta_comp = float(comp[pred] - comp[1 - pred])
    return delta_ref, delta_suff, delta_comp

def faithful_verdict(s, mask_set):
    """Binary ERASER verdict. Returns 0 or 1."""
    delta_ref, delta_suff, delta_comp = _deltas(s, mask_set)
    return int((delta_suff >= ALPHA_SUFF * delta_ref) and ((delta_ref - delta_comp) >= TAU_COMP * delta_ref))

def soft_faithful(s, mask_set):
    """Continuous faithfulness in [0, 1]. 1.0 if the mask passes both ERASER
    thresholds, partial credit below."""
    delta_ref, delta_suff, delta_comp = _deltas(s, mask_set)
    if delta_ref <= 0:
        return 0.0
    suff_score = min(max(delta_suff / delta_ref, 0.0) / ALPHA_SUFF, 1.0)
    comp_score = min(max((delta_ref - delta_comp) / delta_ref, 0.0) / TAU_COMP, 1.0)
    return suff_score * comp_score

def jaccard_distance(a, b):
    a, b = set(a), set(b)
    union = a | b
    if not union:
        return 0.0
    return 1.0 - len(a & b) / len(union)

## **Occlusion helper**

Occlusion is the causal-effect estimate the probe rewards. We ship it precomputed so a first submission needs no forward passes: `occlusion_pack.json` maps `str(sentence_id) -> {delta_ref, deltas, T, cap, pred}` where `deltas[str(tok)]` is the drop in the predicted-class margin when *only* token `tok` is masked. Bigger delta = more the model relies on that token.

`occlusion_scores(s)` returns a `{tok: delta}` dict over the sentence's content tokens. If you want to compute your own attribution (attention rollout, gradient x input) for 3, you have `model` loaded in eager mode. This pack is just the occlusion channel done for you.

In [7]:
with open(DATA / 'occlusion_pack.json') as f:
    OCC_PACK = json.load(f)

def occlusion_scores(s: dict) -> dict:
    """{tok: occlusion_delta} over content tokens, from the precomputed pack."""
    entry = OCC_PACK[str(s['sentence_id'])]
    deltas = {int(k): float(v) for k, v in entry['deltas'].items()}
    return {t: deltas.get(t, 0.0) for t in content_tokens(s)}

def occlusion_ranked(s: dict) -> list:
    """Content tokens sorted by descending occlusion delta (most causal first)."""
    scores = occlusion_scores(s)
    return sorted(content_tokens(s), key=lambda t: -scores[t])

# Sanity check on sentence 0.
_s0 = sentences[0]
print('sentence 0 top-5 occlusion tokens:', occlusion_ranked(_s0)[:5])
print('delta_ref:', OCC_PACK['0']['delta_ref'], '| cap:', cap_for_sentence(_s0))

sentence 0 top-5 occlusion tokens: [20, 17, 22, 9, 14]
delta_ref: 8.073 | cap: 7


## **Submission writers (native CSV)**

One writer per subtask, each emitting `subtaskID,datapointID,answer` with one row per content token. You pass in masks as plain dicts and the writer expands them to per-token binary rows with the right `datapointID` encoding. `subtaskID` is `1` for standalone submissions. Call the matching writer at the end of each subtask's cell.

- `write_1_csv({sid: mask})` : `datapointID = sid*1000 + tok`.
- `write_2_csv({sid: (maskA, maskB)})` : `datapointID = sid*2000 + slot*1000 + tok`.
- `write_3_csv({sid: mask})` : same encoding as 1.

In [8]:
Path('out').mkdir(exist_ok=True)
SUBTASK_ID = 1  # standalone per-part submissions use subtaskID = 1

def write_1_csv(masks_by_sid, path='out/submission_1.csv'):
    with open(path, 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['subtaskID', 'datapointID', 'answer'])
        for s in sentences:
            sid = s['sentence_id']; mset = set(masks_by_sid.get(sid, []))
            print(mset)
            for t in content_tokens(s):
                w.writerow([SUBTASK_ID, sid * 1000 + t, int(t in mset)])

def write_2_csv(pairs_by_sid, path='out/submission_2.csv'):
    with open(path, 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['subtaskID', 'datapointID', 'answer'])
        for s in sentences:
            sid = s['sentence_id']; a, b = pairs_by_sid.get(sid, ([], []))
            for slot, mset in ((0, set(a)), (1, set(b))):
                for t in content_tokens(s):
                    w.writerow([SUBTASK_ID, sid * 2000 + slot * 1000 + t, int(t in mset)])

def write_3_csv(masks_by_sid, path='out/submission_3.csv'):
    with open(path, 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['subtaskID', 'datapointID', 'answer'])
        for s in sentences:
            sid = s['sentence_id']; mset = set(masks_by_sid.get(sid, []))
            for t in content_tokens(s):
                w.writerow([SUBTASK_ID, sid * 1000 + t, int(t in mset)])

## **1, The first colleague (Witness)**

Nominate, for each of the 100 sentences, the small set of tokens that drove the classifier's prediction. Cap: 12 tokens or 30 percent of the real-token count, whichever is smaller. Fewer is fine.

The hidden reference is the **minimal top-occlusion mask that passes the real ERASER probe**, one per sentence. The judge scores per-token **binary F1** of your mask against that reference, so your goal is to reproduce a small faithful mask. `faithful_verdict` is your honest local estimator of *whether a mask is faithful*. A mask that passes it is the kind of mask the reference is made of.

The cell below ships a **simple occlusion-top-half baseline** (no probe check, no greedy stopping). It's decent but leaves points on the table.

In [9]:
a = np.array([1, 2, 3])
a = {}
print_dir(a)

clear
copy
fromkeys
get
items
keys
pop
popitem
setdefault
update
values


In [10]:
# import torch.nn as nn
# # print_dir(tok)
# # print(sentences[0])
# s = sentences[0]
# # print(s["text"])
# # print(am)

# am = sentences[0]["attention_mask"]
# ids = torch.tensor([s['token_ids']], device=DEVICE)
# n_tok = round(min(MAX_MASK_TOKENS, (1 - MAX_MASK_FRACTION) * ids.shape[1]))
# print(n_tok)
# print(tok.decode(ids))
# am = torch.tensor([s['attention_mask']], device=DEVICE)
# pred = s['predicted_label']
# ref = forward_logits(ids, am)
# print(ref)
# delta_ref = max(ref) - min(ref)
# print(delta_ref)
# final = {}
# for i in range(len(sentences)):
#     s = sentences[i]
#     am = torch.tensor([s['attention_mask']], device=DEVICE)
#     ids = torch.tensor([s['token_ids']], device=DEVICE)
#     n_tok = round(min(MAX_MASK_TOKENS, MAX_MASK_FRACTION * ids.shape[1]))
#     masks = []
#     accs = []
#     for j in range(ids.shape[1]):
#         idsc = ids.clone()
#         # print(idsc.shape)
#         idsc[0][j] = 103
#         # print(tok.decode(idsc))
#         out = forward_logits(idsc, am)
#         delta_ref = max(out) - min(out)
#         accs.append(delta_ref.cpu().item())
    
    
#     accs = np.array(accs)
#     idxs = np.argsort(accs)[:n_tok]
#     ids[0][idxs] = 103
#     if i == 0:
#         print(accs)
#         print(ids)
#         print(accs[idxs])
#     # final[s['sentence_id']:ids] 


# # loss = nn.CrossEntropyLoss()(ref, torch.Tensor(1).to(DEVICE) if out[0]>out[1] else torch.Tensor(0).to(DEVICE))
# # ----------------------------------------------------------------------------------------------------------------------------------------------------

# print(out)

In [11]:
print(67)

# ============================================================================
# YOUR CODE HERE  --  Subtask 1: Witness
# ----------------------------------------------------------------------------
# Return one mask (a list of token indices) per sentence. The judge scores
# per-token F1 vs a hidden faithful reference. `faithful_verdict(s, mask)` is
# your local check for whether a mask is faithful. The baseline below is a
# runnable starting point: the single most causal token by occlusion.
# ============================================================================
print(67)
# out = forward_logits(sentences[0], am)

def my_1_mask(s):
    ranked = occlusion_ranked(s)
    print((sorted(ranked[:1])))
    return sorted(ranked[:1])

def my_mask_1_v2(s):
    am = torch.tensor([s['attention_mask']], device=DEVICE)
    ids = torch.tensor([s['token_ids']], device=DEVICE)
    shape = ids.shape[1]
    
    n_tok = round(min(MAX_MASK_TOKENS, (MAX_MASK_FRACTION) * ids.shape[1]))
    n_tok = ids.shape[1] - n_tok
    # print(ids.shape[1], n_tok, MAX_MASK_TOKENS)
    masks = []
    accs = []
    for j in range(ids.shape[1]):
        idsc = ids.clone()
        # print(idsc.shape)
        idsc[0][j] = 103
        # print(tok.decode(idsc))
        out = forward_logits(idsc, am)
        delta_ref = max(out) - min(out)
        accs.append(delta_ref.cpu().item())
    
    
    accs = np.array(accs)
    idxs = np.argsort(accs)[:n_tok]
    ids[0][idxs] = 103
    ids = ids[ids!=103]
    # print(ids)
    ids = ids.tolist()
    
    return ids[1:shape-1]

print(67)
s = sentences[0]
# print(f"{my_mask_1_v2(s)}")sentences

masks_1 = {s['sentence_id']: f"{my_1_mask(s)}" for s in sentences}
# print(masks_1.values())
write_1_csv(masks_1)
print(f'Wrote out/submission_1.csv')


67
67
67
[20]
[14]
[3]
[19]
[16]
[13]
[16]
[15]
[17]
[3]
[19]
[13]
[6]
[19]
[5]
[1]
[22]
[4]
[31]
[3]
[3]
[17]
[7]
[3]
[14]
[14]
[18]
[10]
[30]
[19]
[21]
[7]
[2]
[6]
[4]
[3]
[16]
[2]
[5]
[10]
[7]
[6]
[26]
[7]
[9]
[7]
[31]
[6]
[18]
[1]
[1]
[7]
[15]
[12]
[8]
[11]
[30]
[6]
[8]
[15]
[21]
[9]
[5]
[2]
[11]
[14]
[4]
[5]
[8]
[5]
[2]
[3]
[4]
[4]
[16]
[25]
[12]
[27]
[36]
[15]
[11]
[1]
[1]
[38]
[2]
[5]
[29]
[17]
[4]
[14]
[6]
[30]
[3]
[14]
[24]
[5]
[16]
[4]
[36]
[15]
{']', '0', '[', '2'}
{']', '4', '[', '1'}
{']', '3', '['}
{'9', ']', '[', '1'}
{'6', ']', '[', '1'}
{']', '3', '[', '1'}
{'6', ']', '[', '1'}
{'5', ']', '[', '1'}
{'7', ']', '[', '1'}
{']', '3', '['}
{'9', ']', '[', '1'}
{']', '3', '[', '1'}
{'6', ']', '['}
{'9', ']', '[', '1'}
{'5', ']', '['}
{']', '[', '1'}
{']', '[', '2'}
{']', '4', '['}
{']', '3', '[', '1'}
{']', '3', '['}
{']', '3', '['}
{'7', ']', '[', '1'}
{'7', ']', '['}
{']', '3', '['}
{']', '4', '[', '1'}
{']', '4', '[', '1'}
{'8', ']', '[', '1'}
{']', '0', '[', '1'}
{']', '

In [12]:
sentences[80]["sentence_id"]

80

In [13]:
# ---- Self-check for subtask 1: fraction of masks that pass the ERASER probe --------
# This is the true faithfulness signal. The live judge scores F1 vs a hidden
# faithful reference, but a mask that passes faithful_verdict is exactly the
# kind of mask that reference is made of, so a high pass-rate is what you want.
# (Runs the model over the 100 sentences; needs a GPU, ~a minute.)
passed = [faithful_verdict(s, set(masks_1[s['sentence_id']])) for s in sentences]
print(passed)
frac = float(np.mean(passed))
sizes = [len(masks_1[s['sentence_id']]) for s in sentences]
print(f'1 faithful-mask pass rate: {100 * frac:.1f}%  '
      f'(mean mask size {np.mean(sizes):.1f}, cap-respecting)')
print('Aim to push this toward ~90%+ with minimal masks; that tracks the F1 score.')

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
1 faithful-mask pass rate: 0.0%  (mean mask size 3.5, cap-respecting)
Aim to push this toward ~90%+ with minimal masks; that tracks the F1 score.


## **2, The second colleague (Counter-argument)**

Two distinct rationales per sentence: slot 0 and slot 1, each its own mask, same cap as 1. Per sentence the score is (roughly) $f^A \cdot f^B \cdot d$, where $f^A, f^B$ reward each slot matching a faithful reference and $d$ is the Jaccard distance between your two masks. The winning play is **two disjoint masks that are each individually faithful**: same mask twice gives $d = 0$. One faithful + one junk mask gives a low product.

The baseline below just splits the occlusion ranking into two interleaved halves (disjoint, but not probe-gated), which scores modestly.

In [14]:
# ============================================================================
# YOUR CODE HERE  --  Subtask 2: Counter-argument
# ----------------------------------------------------------------------------
# Return two masks (a, b) per sentence, slot 0 and slot 1. The score rewards
# two disjoint masks that are each faithful. Check them with faithful_verdict
# and jaccard_distance. The baseline below is a runnable starting point.
# ============================================================================

def my_2_masks(s):
    ranked = occlusion_ranked(s)
    cap = cap_for_sentence(s)
    a = sorted(ranked[0:2 * cap:2][:cap])
    b = sorted(ranked[1:2 * cap:2][:cap])
    return a, b

pairs_2 = {s['sentence_id']: my_2_masks(s) for s in sentences}
write_2_csv(pairs_2)
print(f'Wrote out/submission_2.csv')


Wrote out/submission_2.csv


In [15]:
# ---- Self-check for subtask 2: both slots faithful AND masks diverse ---------------
# The score is a product of (faithful A) x (faithful B) x (Jaccard distance).
# You want the fraction of sentences where BOTH masks pass the probe to be
# high AND the mean Jaccard distance to be near 1 (disjoint). (Needs a GPU.)
both_faithful, jaccs, combined = [], [], []
for s in sentences:
    a, b = pairs_2[s['sentence_id']]
    fa = faithful_verdict(s, set(a)); fb = faithful_verdict(s, set(b))
    d = jaccard_distance(a, b)
    both_faithful.append(int(fa and fb)); jaccs.append(d)
    # soft product mirrors the graded quantity for a local estimate
    combined.append(soft_faithful(s, set(a)) * soft_faithful(s, set(b)) * d)
print(f'2 both-slots-faithful: {100 * np.mean(both_faithful):.1f}%  |  '
      f'mean Jaccard distance: {np.mean(jaccs):.2f}')
print(f'2 estimated mean score (soft_f_A * soft_f_B * d): {np.mean(combined):.3f}  '
      f'(higher is better; probe-gated disjoint pairs win)')

2 both-slots-faithful: 21.0%  |  mean Jaccard distance: 1.00
2 estimated mean score (soft_f_A * soft_f_B * d): 0.462  (higher is better; probe-gated disjoint pairs win)


## **3, The third colleague (Journalist)**

The journalist doesn't run the model. For each sentence she has a hidden human-consensus rationale set, built by running **three attribution methods (attention rollout, occlusion, gradient x input)**, taking each method's top-k tokens, and keeping any token that at least **two of the three** methods rank in their top-k. Roughly 4-8 tokens per sentence, mostly the sentiment-bearing words. She scores you per token with **binary F1** against that hidden set.

The protocol is published on purpose: contestants whose strategy *matches her construction* do well. Note the trap: **occlusion alone (which wins 1) only partly matches the consensus.** The mask that's most faithful to the model is not the same as the mask a panel of methods agrees on. To do well here you need to build all three attributions and take the 2-of-3 vote, not just reuse your 1 mask.

The baseline below is occlusion-top-6, the single-method shortcut. It scores a mediocre F1 on the practice set, which is the point: it shows you how much the missing two methods are worth.

### How to approach 3

You already have occlusion. The hidden gold is a 2-of-3 vote across three attribution methods, so you need two more, then the vote.

- **Attention rollout.** The model's own attention weights show where it attends. Load it so it returns them (`attn_implementation='eager'`, then call with `output_attentions=True`), and aggregate across heads to score each token from the `[CLS]` row.
- **Gradient times input.** How much each token contributes, from the gradient of the predicted-class logit with respect to the input embeddings, times the embeddings, summed over the hidden dimension.

Take each method's top-k tokens (k around 6), keep the tokens that at least two of the three agree on, and respect the cap. The practice set was built with this exact protocol, so use `practice_rationales.json` to calibrate before you submit.

In [16]:
# ============================================================================
# YOUR CODE HERE  --  Subtask 3: Journalist
# ----------------------------------------------------------------------------
# Return one mask per sentence. The judge computes per-token F1 vs a hidden
# human-consensus set: a 2-of-3 vote across three attribution methods, each
# top-k. The baseline below uses a single occlusion token, which only partly matches.
# ============================================================================

def my_3_mask(s):
    ranked = occlusion_ranked(s)
    return sorted(ranked[:1])

masks_3 = {s['sentence_id']: my_mask_1_v2(s) for s in sentences}
write_3_csv(masks_3)
print(f'Wrote out/submission_3.csv')


Wrote out/submission_3.csv


In [17]:
# ---- Self-check for subtask 3: token-F1 on the labeled PRACTICE set ----------------
# 25 practice sentences ship with exact rationale labels built by the SAME
# published protocol as the hidden GT, so this F1 is a faithful proxy for your
# live 3 score. They also ship their own occlusion pack so the baseline runs
# with no forward passes. Apply YOUR 3 strategy to these sentences.
from sklearn.metrics import f1_score as _f1

_psent = json.load(open(DATA / 'practice_sentences.json'))
_prat = json.load(open(DATA / 'practice_rationales.json'))
_pocc = json.load(open(DATA / 'practice_occlusion_pack.json'))

def _practice_occ_scores(s):
    e = _pocc[str(s['sentence_id'])]
    d = {int(k): float(v) for k, v in e['deltas'].items()}
    return {t: d.get(t, 0.0) for t in content_tokens(s)}

def my_3_mask_practice(s):
    # Mirror my_3_mask, but read the practice occlusion pack. If your real
    # strategy uses the model (attention/grad), call it here on `s` directly.
    scores = _practice_occ_scores(s)
    cap = cap_for_sentence(s)
    ranked = sorted(content_tokens(s), key=lambda t: -scores[t])
    return sorted(ranked[:min(6, cap)])

_yt, _yp = [], []
for s in _psent:
    mask = set(my_3_mask_practice(s))
    truth = set(_prat[str(s['sentence_id'])])
    for i in content_tokens(s):
        _yt.append(int(i in truth)); _yp.append(int(i in mask))
_f1v = _f1(_yt, _yp, pos_label=1, zero_division=0)
_FLOOR, _CEIL = 0.24, 0.94  # the judge's F1 -> score normalization for 3
_est = int(round(100 * max(0.0, min(1.0, (_f1v - _FLOOR) / (_CEIL - _FLOOR)))))
print(f'3 practice token-F1: {_f1v:.3f}  ->  estimated 3 score ~{_est} '
      f'(judge floor {_FLOOR}, ceil {_CEIL})')
print('Occlusion-only tops out here; adding attention rollout + grad x input '
      'and taking the 2-of-3 vote is what closes the gap.')

3 practice token-F1: 0.601  ->  estimated 3 score ~52 (judge floor 0.24, ceil 0.94)
Occlusion-only tops out here; adding attention rollout + grad x input and taking the 2-of-3 vote is what closes the gap.


---

## **Before you submit**

- Three CSVs land in `out/`: `submission_1.csv`, `submission_2.csv`, `submission_3.csv`, each `subtaskID,datapointID,answer` with one binary row per content token.
- Every mask respects `cap_for_sentence`; `[CLS]` (index 0) and the final `[SEP]` are never nominated.
- Subtask 1: maximize the `faithful_verdict` pass rate with minimal masks. Subtask 2: two disjoint masks that both pass the probe. Subtask 3: match the 2-of-3 attribution consensus, not just occlusion.

In [18]:
# ---- Combine the three parts into ONE submission for the folded task ----
# The task is a single task with three subtasks (1, 2, 3) at
# subtaskID 1, 2, 3. Upload the ONE combined file written below, not the three
# per-part files.
import csv
_parts = [("out/submission_1.csv", 1), ("out/submission_2.csv", 2), ("out/submission_3.csv", 3)]
with open("out/submission.csv", "w", newline="") as _f:
    _w = csv.writer(_f); _w.writerow(["subtaskID", "datapointID", "answer"])
    for _path, _sid in _parts:
        for _row in Path(_path).read_text().splitlines()[1:]:
            _, _dp, _ans = _row.split(",", 2)
            _w.writerow([_sid, _dp, _ans])
print("Wrote out/submission.csv  (subtaskID 1/2/3).  Upload THIS single file.")

Wrote out/submission.csv  (subtaskID 1/2/3).  Upload THIS single file.
